# Working advection-diffusion model in Arctic basin

### Based on gridap Tutorial 10: Advection-diffusion

twnh Jun '25

## Problem statement

Formally, a steady-state advection-diffusion equation with Dirichlet boundary conditions could be defined as:

$ \nabla \cdot \left ( D\nabla T - \vec{V}T  \right ) =0 $ ,\
$ T = g $ on boundaries,

where:
   -  $T$ is the scalar quantity being transported (here T refers to temperature);
   -  $\vec{V}$ is the velocity vector of the fluid (representing advection);
   -  $D$ is the diffusion coefficient (a constant that characterizes the rate of diffusion).

## Numerical Scheme

The weak form of a PDE is a reformulation that allows the problem to be solved in a broader
function space. Instead of requiring the solution to satisfy the PDE at every point (as in the strong form),
the weak form requires the solution to satisfy an integral equation. This makes it particularly suitable for
numerical methods such as the Finite Element Method.

Since we already have the original PDE (strong form), we can multiply each side by a test function $v \in H^1(\Omega)$ (functions with square-integrable first derivatives). $v$ satisfies the Dirichlet boundary condition of $0$. Make integral on both sides.

The weak form associated with this formulation is, find $T \in H^1(\Omega)$, such that:
$$
\int_\Omega v (\mathbf{u} \cdot \nabla T) \, \mathrm{d}\Omega
+ \int_\Omega D \nabla T \cdot \nabla v \, \mathrm{d}\Omega
= 0
$$

## FE spaces

In [1]:
using Gridap
using GridapGmsh

Import the model from file using GmshDiscreteModel:

In [2]:
model_basin = GmshDiscreteModel("ArcticBasin.msh")
# model_basin = GmshDiscreteModel("pentagon_mesh.msh")
# writevtk(model_basin,"ArcticBasin")
# labels = get_face_labeling(model_basin)

Info    : Reading 'ArcticBasin.msh'...
Info    : 40 entities
Info    : 3882 nodes
Info    : 7790 elements
Info    : Done reading 'ArcticBasin.msh'


UnstructuredDiscreteModel()

Set up the test FE space $V0$, which conforms the zero value boundary conditions.

In [3]:
order = 1
reffe = ReferenceFE(lagrangian,Float64,order)

# V0 = TestFESpace(model_basin,reffe;conformity=:H1,dirichlet_tags=["l1","l2","l3","l4","l5"])
# V0 = TestFESpace(model_basin,reffe;conformity=:H1,dirichlet_tags=["FresheningPatch","WarmingPatch"])
V0 = TestFESpace(model_basin,reffe;conformity=:H1,dirichlet_tags=["Top","Bottom", "WarmingPatch"])

UnconstrainedFESpace()

Set up the boundary conditions. Define the trial space $Ug$.

In [4]:
# boundary_cond = [200,100,0,0,100];
boundary_cond = [1, -1, 9];
Ug = TrialFESpace(V0,boundary_cond)

TrialFESpace()

Do the integration over domain $\Omega$.

In [5]:
degree = 2
Ω = Triangulation(model_basin)
dΩ = Measure(Ω,degree)

GenericMeasure()

## Weak form

Define the velocity field $\vec{V} = (u_1,u_2, u_3)$. 

In [6]:
khat = VectorValue(0.0, 0.0, 1.0)
velocity = VectorValue(0.0, 0.0, 0.0);

Duffusion coefficient is defined as:

In [7]:
D = 1.0;

The weak form can thus be represented as:

In [8]:
res(u, v) = ∫(v * (velocity ⋅∇(u)) + ∇(v) ⋅ (D * ∇(u))) * dΩ
b(v) = 0.0

b (generic function with 1 method)

## Solution

Now build the FE problem and use the solver. This nonlinear solver is based on https://gridap.github.io/Tutorials/dev/pages/t004_p_laplacian/

In [9]:
using LineSearches: BackTracking
nls = NLSolver(show_trace=true, method=:newton, linesearch=BackTracking())
solver = FESolver(nls)

import Random
Random.seed!(1234)
x = rand(Float64,num_free_dofs(Ug))
uh0 = FEFunction(Ug,x)
op = FEOperator(res,Ug,V0)
uh, = solve!(uh0,solver,op)

Iter     f(x) inf-norm    Step 2-norm 
------   --------------   --------------
     0     1.317842e+01              NaN
     1     1.021405e-14     2.545217e+03


(SingleFieldFEFunction(), NLSolversCache())

These linear solvers work too:

In [10]:
# op = AffineFEOperator(res,b,Ug,V0)
# ls = LUSolver()
# solver = LinearFESolver(ls)

# uh = Gridap.Algebra.solve(op)
# # uh = solve(solver,op)

Ouput the result as a `.vtk` file.

In [11]:
writevtk(Ω,"AdvectionDiffusionTest",cellfields=["uh"=>uh])

(["results.vtu"],)